# A hyperspectral ROI picker 

**This notebook explores the possibilities of building a four-pane hyper spectral cube dashboard.**

**My latest attempts to get param to listen to changes in the holoviews BOxEdit stream have failed. Not sure how to fix this yet.**

## Data now 

In [ ]:
from fairdatanow import data_now # Hoi Koen, zie voor code mijn notebook in fairdatanow-nbs 
from hyperviz import roi_picker, rasterize 
import hyperviz as vz 
import holoviews as hv 
from holoviews import opts, RGB, streams 
import toml 
import numpy as np 
import pandas as pd 
import ipynb_path

In [ ]:
ipynb_path.get()

In [ ]:
hv.extension('bokeh') 

In [ ]:
from holoviews.operation.datashader import rasterize 
import holoviews as hv 
from holoviews import opts, RGB, streams 
import panel as pn 

import numpy as np 
import toml 
import re 
import os 

colors = hv.Cycle.default_cycles['default_colors']
boxes_styles = dict(fill_alpha=0.1, line_color='red')
style_opts = dict(size=10, fill_color=colors, line_color='green')
frame_width = 300 
opts.defaults(opts.RGB(aspect='equal', frame_width=frame_width, invert_yaxis=False), 
              opts.Image(aspect='equal', frame_width=frame_width), 
              opts.Curve(frame_width=frame_width, frame_height=frame_width, color=hv.Cycle(colors)), 
              opts.Labels(text_color='red'), 
              opts.Rectangles(fill_color=None, line_color='green'), 
              opts.Polygons(fill_color=None, line_color='purple'))    

In [ ]:
toml_txt = '''
# EXPLORING PIGMENT SPECTRAL DATA 

# Ten selected ukiyo-e prints have been transported to Amsterdam for further analysis 
# Question is how we can identify the colorants that have been used in these prints. 
# Starting point is the set of 'super white' calibrated hyper spectral data cubes 
# prepared by Tessa and Gauthier and stored as numpy npz files.  

# object numbers for the ten 
prints_in_Amsterdam = [
 'RV-1-4468-544',
 'RV-1-4469-484',
 'RV-1-4469-58',
 'RV-1-4469-x6',
 'RV-1-4469Q',
 'RV-1-4470-12',
 'RV-1-4470-27',
 'RV-360-2345g',
 'RV-360-2359-2',
 'RV-360-6886']

# here are the 10 corresponding spectral data cubes processed by Gauthier and Tessa 
[nc_adam10_cubes] 
RV-1-4468-544 = ".*RIS/interim/.*RV-1-4468-544.*[.]npz"
RV-1-4469-484 = ".*RIS/interim/.*RV-1-4469-484.*[.]npz"
RV-1-4469-58 = ".*RIS/interim/.*RV-1-4469-58.*[.]npz"
RV-1-4469-x6 = ".*RIS/interim/.*RV-1-4469-x6.*[.]npz"
RV-1-4469Q = ".*RIS/interim/.*RV-1-4469Q.*[.]npz"
RV-1-4470-12 = ".*RIS/interim/.*RV-1-4470-12.*[.]npz"
RV-1-4470-27 = ".*RIS/interim/.*RV-1-4470-27.*[.]npz"
RV-360-2345g = ".*RIS/interim/.*RV-360-2345g.*[.]npz"
RV-360-2359-2 = ".*RIS/interim/.*RV-360-2359-2.*[.]npz"
RV-360-6886 = ".*RIS/interim/.*RV-360-6886.*[.]npz"


# here are the corresponding high res tifs 
[nc_tifs] 
# regexes for the tifs 
RV-1-4468-544 = ".*akama.*RV-1-4468-544[.]tif"
RV-1-4469-484 = ".*akama.*RV-1-4469-484[.]tif"
RV-1-4469-58 = ".*akama.*RV-1-4469-58[.]tif"
RV-1-4469-x6 = ".*akama.*RV-1-4469-x6[.]tif"
RV-1-4469Q = ".*akama.*RV-1-4469Q[.]tif"
RV-1-4470-12 = ".*akama.*RV-1-4470-12[.]tif"
RV-1-4470-27 = ".*akama.*1-4470-27[.]tif"      # TIF NAME WITHOUT RV prefix!  
RV-360-2345g = ".*akama.*RV-360-2345-?g[.]tif"
RV-360-2359-2 = ".*akama.*RV-360-2359-2[.]tif"
RV-360-6886 = ".*akama.*RV-360-6886[.]tif" 

# In order to identify colorants on the prints we will also need spectral data for reference samples
# Perhaps incomple but a start would be to take a look at the Boston Museum of Fine Arts samples  
# here are 19 colorant plus a paper background spectra all measured with TIDAS spectrometer 

# nineteen colorants 
[nc_boston_colorants] 
blue_dayflower = ".*RS/processed/.*blue_dayflower.*[.]txt"
blue_indigo = ".*RS/processed/.*blue_indigo.*[.]txt"
blue_prussianblue = ".*RS/processed/.*blue_prussianblue.*[.]txt"
bronze_brasspowder = ".*RS/processed/.*bronze_brasspowder.*[.]txt"
brown_ochre = ".*RS/processed/.*brown_ochre.*[.]txt"
green_indigo-orpiment = ".*RS/processed/.*green_indigo-orpiment.*[.]txt"
pink_safflower = ".*RS/processed/.*pink_safflower.*[.]txt"
purple_safflower-dayflower = ".*RS/processed/.*purple_safflower-dayflower.*[.]txt"
red_cochineal = ".*RS/processed/.*red_cochineal.*[.]txt"
red_ironoxide = ".*RS/processed/.*red_ironoxide.*[.]txt"
red_redlead = ".*RS/processed/.*red_redlead.*[.]txt"
red_sappenwood = ".*RS/processed/.*red_sappenwood.*[.]txt"
red_vermilion = ".*RS/processed/.*red_vermilion.*[.]txt"
white_leadwhite = ".*RS/processed/.*white_leadwhite.*[.]txt"
white_mica = ".*RS/processed/.*white_mica.*[.]txt"
yellow_orpiment = ".*RS/processed/.*yellow_orpiment.*[.]txt" # removed a redundant spectrum from nc with Gauthier 
yellow_tumeric-orpiment = ".*RS/processed/.*yellow_tumeric-orpiment.*[.]txt"
yellow_tumeric = ".*RS/processed/.*yellow_tumeric.*[.]txt"
yellow_yellowwood = ".*RS/processed/.*yellow_yellowwood.*[.]txt"

# In order to calculate absorbance spectra we need the reflectance of the background paper 
# of the Boston samples 
[nc_boston_paper] 
cream_paper = ".*RS/processed/.*cream_paper.*[.]txt"

[nc_boston_overviews]
overviews_x4 = ".*external/rma.*overview_monsters.*[.]jpeg" 
'''

In [ ]:
url = 'https://laboppad.nl/ukiyo-e-world'
toml_dict = toml.loads(toml_txt)
obj_nums = toml_dict['prints_in_Amsterdam']
npz_files_dict = data_now(url, toml_txt, nc_key='nc_adam10_cubes', verbose=False)
tif_files_dict = data_now(url, toml_txt, nc_key='nc_tifs', verbose=False)
colorants_files_dict = data_now(url, toml_txt, nc_key='nc_boston_colorants', verbose=False)
paper_files_dict = data_now(url, toml_txt, nc_key='nc_boston_paper', verbose=False)
overviews_files_dict = data_now(url, toml_txt, nc_key='nc_boston_overviews', verbose=False) 

In [ ]:
toml_

## Ten ukiyo-e cubes

In [ ]:
rgbx10_views = [rasterize(hv.RGB.load_image(tif_files_dict[num][0]).opts(title=f'[{i}] {num}', shared_axes=False, aspect='equal')) for i, num in enumerate(obj_nums)]
rgbx10_layout = hv.Layout(rgbx10_views)

In [ ]:
rgbx10_layout.cols(5)

Let's pick a number... 

In [ ]:
n = 6
num = obj_nums[n]
npz = np.load(npz_files_dict[num][0])
cube = npz['image'][:,:, ::-1].transpose(1, 2, 0)
wavelengths = npz['wavelengths'] 
h, w, d = cube.shape
bounds = [0, 0, w, h]
tif_file = tif_files_dict[num][0]
pseudo_rgb = cube[:,:, [70, 53, 19]] 

## A four panes cube dashboard 

**2026/07/15** 

Having succeeded to create an bi-directional ROI Annotation editor I now want to try to combine four panes: 
1) a high res tif viewer,
2) a pseudo color cube viewer with BoxEdit stream and dynamic map labels overlay,
3) a ROI spectrum Curves dynamic map,
4) a ROI annotation editor. 

My guess is that I should combine these in a panel Viewer. Question is if this can be combined with holoviews? 

In [ ]:
from holoviews.operation.datashader import rasterize 
import holoviews as hv 
from holoviews import opts, RGB, streams 
import panel as pn 
import param 
import numpy as np 

from panel.viewable import Viewer 
pn.extension('codeeditor')
hv.extension('bokeh')

In [ ]:
class CubeDashboard(Viewer): 

    cube = param.Array()
    tif_file = param.String() 
    bounds = param.List()
    roi_labels = param.String(default='Hi there') 
    roi_xy_dict = param.Dict(default={'x0':[200], 'y0':[200], 'x1': [250], 'y1':[250]}, allow_refs=True) 
    
    #xylabels_list = param.List()


    # not sure here how to deal with non param stuff 
    # I believe it is better to move this to init
    #boxes = hv.Rectangles([(50, 50, 100, 100)]).opts(line_color='red', fill_color=None)  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
    #boxes_stream = hv.streams.BoxEdit(source=boxes) 
    #boxes_stream.add_subscriber(on_boxedit) 


    def __init__(self, **params): 
        super().__init__(**params) 

        # not sure here how to deal with non param stuff 
        self.boxes = hv.Rectangles([(50, 50, 100, 100)]).opts(line_color='red', fill_color=None)  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
        self.boxes_stream = hv.streams.BoxEdit(source=self.boxes, data=self.roi_xy_dict) #{'x0':[], 'y0':[], 'x1': [], 'y0':[]})

        # the problem here is that this seems to overwrite and not create a reactive parameter 
        self.roi_xy_dict = self.boxes_stream.param.data 

        # watchers should come here 
        self.param.watch(self.add_newline, ['roi_labels'], queued=True)

        # I think we need to watch both changes in the roi_labels string 
        # and somehow the boxes_stream to update the xylabels_list 

        # not clear to me now why this does not work 
        # let's try param.depend(watch=True)
        #self.param.watch(self.labelize_rois,  ['roi_labels', 'roi_xy_dict'], queued=True)

    @param.depends('cube', watch=True, on_init=True)
    def get_pseudo_rgb(self): 
        self.pseudo_rgb = self.cube[:,:, [70, 53, 19]] 
        h, w, d = self.cube.shape
        self.bounds = [0, 0, w, h]    

    # triggered by watcher 
    def add_newline(self, event):  
        if (not self.roi_labels.endswith('\n')): 
            self.roi_labels = f'{self.roi_labels}\n' 


    #def on_boxedit(self, data): 
    #    print('Invoked boxedit') 

    @param.depends('roi_labels', 'roi_xy_dict', watch=True)  
    def labelize_rois(self): #, event): 
        print('You edited the ROI labels or boxes') 



    def __panel__(self): 

        # pane 1 
        tif_view = rasterize(RGB.load_image(self.tif_file, bounds=self.bounds))
        
        # pane 2 
        cube_view = rasterize(RGB(self.pseudo_rgb, bounds=self.bounds))
        #labels_view = hv.Labels(self.param.labels_list)
        labels_view = hv.Labels([(100, 100, 'hi')]) 
        # can we add a dynamic map here with a parametrized callback? 
        cube_overlay = hv.Overlay([cube_view, self.boxes, labels_view]).collate() # not clear to me why this is needed. Perhaps due to rasterize? 

        # pane 3 
        title = pn.pane.Str('ROI labels')
        editor = pn.widgets.CodeEditor.from_param(self.param.roi_labels, max_height=100)
        editor_pane = pn.Column(title, editor) 

        # pane 4 
        # ROI spectra here  
        

        return pn.Row(tif_view, cube_overlay, editor_pane)

In [ ]:
cube_viewer = CubeDashboard(cube=cube, tif_file=tif_file)

In [ ]:
cube_viewer

In [ ]:
pn.pane.Str(cube_viewer.boxes_stream.param.data)

In [ ]:
pn.pane.Str(cube_viewer.roi_xy_dict)

# Wanderings


## Another attempt towards a reactive cube dashboard (no ROI annotations yet)

**2026/07/03**  

Let's start very simple with a BoxEdit stream. Without any param reactivity yet... 


In [ ]:
boxes = hv.Rectangles([(0, 0, 100, 100)]).opts(line_color='red', fill_color=None)  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
boxes_stream = hv.streams.BoxEdit(source=boxes)

In [ ]:
boxes

Now we add an underlaying image. 

In [ ]:
rgb_view = hv.RGB(pseudo_rgb, bounds=bounds).opts(aspect='equal')
rgb_view

In [ ]:
rgb_view * boxes

As a next step I need to add dynamic labels with a callback 

In [ ]:
def roi_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple formatted string hack to position labels outside boxes  
        label_list = [[xi, yi, f'    {i}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))]   
        labels = hv.Labels(label_list).opts(text_color='orange')

    return labels 

In [ ]:
labels_dmap = hv.DynamicMap(roi_labelize, streams=[boxes_stream]) 

In [ ]:
labels_dmap

For some reason these labels do not update across cells, but let's see what happens if we overlay them. 

In [ ]:
roi_overlay = rgb_view * boxes * labels_dmap
roi_overlay

**Dynamic labeling does not work across different cells, but within a single overlay this is fine!**

Ok, as a next step, let's try to add the roi spectrum view. But first try to expand the layout. 

In [ ]:
twice = roi_overlay + roi_overlay 
twice

We see here that the behavior of the BoxEdit stream is not simply copied as I expected. Could I have included the boxes_stream as an overlay? Is that even possible? Well, no. This throws an error. 

In [ ]:
#roi_stream_overlay = rgb_view * boxes * labels_dmap * boxes_stream # error

This means I do not fully understand the dynamics of a DynamicMap. Let's take another look at the docs: [Custom-interactivity](https://holoviews.org/user_guide/Custom_Interactivity.html#custom-interactivity) and [using-parameterized-classes-as-a-stream](https://holoviews.org/user_guide/Responding_to_Events.html#using-parameterized-classes-as-a-stream). Mm, both are not much of a help. 

**My understanding is that the BoxEdit menu listens to the specific element that is selected and only draws boxes there. This is perhaps not a problem, but a feature.** What happens if we use panel to create these duplicates: 

In [ ]:
pn.extension()
pn.Row(roi_overlay, roi_overlay)

Ok, so here we see more clearly that the menus for the left and right image aren't the same and do not sync the drawn roi's across them. **Let's say this is a feature and not a bug.**   

Now let's add a dynamic spectrum viewer just like the calcium dataset example. To begin with we need to check if we get a **static** spectrum viewer: 

In [ ]:
h, w, d = cube.shape 
ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 

In [ ]:
data = boxes_stream.data

In [ ]:
curves = {}
rois = zip(data['x0'], data['x1'], data['y0'], data['y1'])
for i, (x0, x1, y0, y1) in enumerate(rois):
    selection = ds.select(x=(x0, x1), y=(y0, y1))
    curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean)).opts(framewise=True)
    
spectrum_view = hv.NdOverlay(curves, kdims='ROI spectra') # static 

In [ ]:
spectrum_view

Ok, nice. This works. The issue that now pops up is that we need the cube dataset. Probably the best way to deal with this is to create a Dataset instance as a global variable outside the callback function. For now.   

In [ ]:
h, w, d = cube.shape 
cube_dataset = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 

Ok, now let's create a dynamic map callback function that just takes the boxes stream roi data as its single argument. 

In [ ]:
def roi_spectra(data):
    '''Dynamic map callback function to compute average spectra for rectangular Regions Of Interest. '''
    
    if not data: # or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([], 'wavelength', 'reflectance')}, kdims='ROI spectra')

    curves = {}
    rois = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(rois):
        selection = cube_dataset.select(x=(x0, x1), y=(y0, y1))
        curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean)).opts(framewise=True)
        
    spectrum_view = hv.NdOverlay(curves, kdims='ROI spectra')

    return spectrum_view        

And create the corresponding dynamic map with the boxes_stream.

In [ ]:
roi_spectra_dmap = hv.DynamicMap(roi_spectra, streams=[boxes_stream])

In [ ]:
roi_spectra_dmap

**This works!!**

If we now bring both holoviews elements together we get this:

In [ ]:
both = roi_overlay + roi_spectra_dmap
both

Next thing is to add a high res tif viewer. 

In [ ]:
highres_view = rasterize(hv.RGB.load_image(tif_file, bounds=bounds)) # add bounds! 
highres_view

In [ ]:
triple = highres_view + roi_overlay + roi_spectra_dmap
triple

Mm, what is next? We need to create the possibility to initialize the dashboard with predefined ROI's in a toml file. I realize now that this is functionality that needs to be explored in the fairdatanow package based on something like tomlkit. Let's explore this now first. 

In [ ]:
roi_coordinates = pn.pane.Str(boxes_stream.param.data)

In [ ]:
pn.Column(triple, roi_coordinates)

In [ ]:
hv.Table(boxes_stream.data, list(boxes_stream.data.keys())).opts(editable=True)

Unfortunately the holoviews [ML Annotators](http://build.holoviews.org/user_guide/Annotators.html) functionality **contains bugs** that make it unusable. So let's go for a dynamic map with an editable Table...  

In [ ]:
label_table = hv.Table({'a': ['1', '2', '3']}, ['a']).opts(editable=True)
label_table

In [ ]:
import datetime as dt
import pandas as pd
import panel as pn

pn.extension()

In [ ]:
df = pd.DataFrame({
    'int': [1, 2, 3],
    'float': [3.14, 6.28, 9.42],
    'str': ['A', 'B', 'C']})

In [ ]:
df_widget = pn.widgets.DataFrame(df)

df_widget

In [ ]:
df_widget.value['str']

In [ ]:
roi_labels = ['apple-green', 'blue-gray', 'red'] 

In [ ]:
def roi_tomlize(cube_id, data, roi_labels=None): 
   
    #roi_arr = pd.DataFrame(data).values 
    #if roi_labels is None: 
    #    roi_labels = [f'ROI_{i:02}' for i in range(len(roi_arr))] 
    #
    #toml_table = f'[{cube_id}] # x0, y0, x1, y1\n'

    #for label, [x0, y0, x1, y1] in zip(roi_labels, roi_arr): 
    #    row = f'{label} = [{x0:0.2f}, {y0:0.2f}, {x1:0.2f}, {y1:0.2f}]\n' 
    #    toml_table += row 

    return str(data)
    

In [ ]:
pn.pane.Str(roi_tomlize(num, roi_labels=None, data=boxes_stream.data))

In [ ]:
roi_dict = boxes_stream.data

In [ ]:
roi_dict['roi_labels'] = roi_labels

In [ ]:
print(toml.dumps(roi_dict))

And now what? We can create a table with another column attached that contains labels. Question is how to detect if a specific roi is deleted? 

In [ ]:
print(toml.dumps({'roi_labels': roi_labels, 'roi_data': boxes_stream.data})) 

In [ ]:
roi_dict_rounded = {}

for k in roi_dict.keys(): 
    roi_dict_rounded[k] = [round(x, 2) for x in roi_dict[k]] 

In [ ]:
roi_dict_rounded

Not sure now how to make this reactive? And how to combine this with adding labels? 

In [ ]:
print(toml.dumps(roi_dict_rounded))

## An advanced cube explorer (not working) 

**I guess I should not put the streams inside the callbacks...** 

Following the final holoviews example of [Declarative Dashboards](https://holoviews.org/user_guide/Dashboards.html#declarative-dashboards) this seems to be the way forward. Created a working toy example here of a reactive flipping image in combination with reactive selecting ROI's. 

**Key insight is to avoid things like overlays as a return value in the parametrized dynamic map callback method. This breaks the reactivity and disables the box edit menu.**  

Next steps are:  

- rename method
- add wavelengths
- labelize 
- create calcium like roi spectrum viewer method from Dataset 
- persistent roi's
- add doc strings 

In [ ]:
import holoviews as hv 
import param 
import panel as pn

class CubeExplorer(param.Parameterized): 
    
    cube_array = param.Array()#allow_refs=True) # need to test if this is crucial 
    cube_wavelengths = param.Array()
    
    boxes_dict = param.Dict() # needed? 

    boxes = hv.Rectangles([(0, 0, 100, 100)]).opts(line_color='red')  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
    boxes_stream = hv.streams.BoxEdit(source=boxes) 
    
    #boxes_dict = boxes_stream.data
    
    
    @param.depends('cube_array', 'cube_wavelengths', 'boxes', 'boxes_stream', 'boxes_dict', watch=True) # reduce 
    def roi_picker(self): 

    # better assert? 
        if self.cube_array is not None: 
            h, w, d = self.cube_array.shape 
            bounds = [0, 0, w, h]  
            pseudo_rgb = self.cube_array[:,:, [70, 53, 19]] 
            rgb_view = hv.RGB(pseudo_rgb, bounds=bounds)  
            rgb_view.opts(aspect='equal')#, framewise=True) 
            
            # probably need convert roi dictionary to toml string? 
            self.boxes_dict = self.boxes_stream.data 
            
            # returning an overlay is probably not a good idea 
            #cube_view = hv.Overlay([rgb_view, self.boxes]) #.opts(tools=['box_edit'])]) # this adds a button but is lame 
            # instead return rgb_view 

        return rgb_view

    @param.depends('boxes', 'boxes_stream', watch=True)
    def roi_labelize(self): 
        '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
 
        if (self.boxes_stream.param.data.rx() is None):# or (self.boxes_stream.param.data.rx.len() < 1):
            labels = hv.Labels([[100, 100, 'hi']])
        else: 
            # simple hack to position labels 
            #label_list = [[xi, yi, f'x    {i}'] for i, [xi, yi] in enumerate(zip(self.boxes_stream.data.rx()['x1'], self.boxes_stream.data.rx()['y0']))]   
            #labels = hv.Labels(label_list)
            labels = hv.Labels([[200, 200, 'bye']])
    
        return labels 
        

explorer = CubeExplorer(cube_array=cube, cube_wavelengths=wavelengths)

cube_dmap = hv.DynamicMap(explorer.roi_picker) 
labels_dmap = hv.DynamicMap(explorer.roi_labelize)
overlay = cube_dmap * explorer.boxes * labels_dmap
overlay

In [ ]:
explorer.

In [ ]:
explorer.cube_array = cube[::-1]

In [ ]:
pn.pane.Str(explorer.boxes_stream.param.data)

This advanced cube explorer is too primitive because inside a dynamic map is does react to updated Parameter values, but it does not allow drawing boxes, and vice versa: 

In [ ]:
explorer.param.cube_array = cube[::-1]

In [ ]:
explorer.boxes_stream.data == None

In [ ]:
explorer.boxes_stream.data['x0']

## A simple cube viewer callback with ParamRefs streams  

See example below. 

In [ ]:
import holoviews as hv 
import param 


class Cubic(param.Parameterized): 
    cube_array = param.Array()

    

# instantiate streams 
cube_6 = Cubic(cube_array=cube) 

boxes = hv.Rectangles([(0, 0, 100, 100)]).opts(line_color='red', fill_color='green', fill_alpha=0.2)  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
boxes_stream = hv.streams.BoxEdit(source=boxes) 

# create parameter references 
refs = {'cube_ref': cube_6.param.cube_array, 'boxes_ref': boxes_stream.param.data}

In [ ]:
def cube_viewer(cube_ref, boxes_ref): 
    '''Callback function for DynamicMap with ''' 
    
    # get pseudo rgb from 
    h, w, d = cube_ref.shape 
    bounds = [0, 0, w, h]  
    pseudo_rgb = self.cube_ref[:,:, [70, 53, 19]] 

    # make viewer 
    rgb_view = hv.RGB(pseudo_rgb, bounds=bounds)  
    rgb_view.opts(aspect='equal') 

    # get 
    boxes_dict = self.boxes_stream.data 

            cube_view = hv.Overlay([rgb_view, self.boxes])

    return cube_view

    

In [ ]:
cube6 = Cube(cube_array=cube)

In [ ]:
cube_array

In [ ]:
cube_array.pa = cube

In [ ]:
refs = {'cube': cube_array.

## Example of a simple dynamic map callback with ParamRefs streams

Let's start by simply copying the dynamic table example: https://holoviews.org/user_guide/Responding_to_Events.html#what-is-a-parameter-reference

In [ ]:
import holoviews as hv 
import param 

class BankAccount(param.Parameterized):
    balance = param.Number(default=0, doc="Bank balance in USD")
    overdraft = param.Number(default=200, doc="Overdraft limit")

janes_account = BankAccount(name='Jane', balance=300)
bobs_account = BankAccount(name='Bob', balance=-200)

investments = 2500
debt = 1800
total = janes_account.param.balance.rx() + bobs_account.param.balance.rx() + investments - debt

refs = {'jane': janes_account.param.balance, 'bob': bobs_account.param.balance, 'total': total}

def bars(jane, bob, total):
    return hv.Bars([('Jane', jane), ('Bob', bob), ('Combined', total)], 'Account', 'Balance ($)').opts(tools=['tap'])

bars_dmap = hv.DynamicMap(bars, streams=[hv.streams.ParamRefs(refs=refs)])
bars_dmap

In [ ]:
bobs_account.balance = 500

Ok, now let's rebuild the cube explorer example, but separate parametrized objects 

In [ ]:
boxes = hv.Rectangles([(0, 0, 100, 100)]).opts(line_color='red', fill_color='green', fill_alpha=0.2)  # do we need polygons here? Would be clearer to use Rectangles or Bounds? 
boxes_stream = hv.streams.BoxEdit(source=boxes)

In [ ]:
#hv.help(hv.Bars)

## Cannot get the roi_picker to work inside a normal function 

Here are the dynamic map callbacks and the bunch of holoviews elements. They work. but as soon as I try to pack them in a normal python function the no longer work... 

**Perhaps the problem here is that my the holoviews cube Dataset is not available in my read_spectra() function!**

https://holoviews.org/gallery/demos/bokeh/box_draw_roi_editor.html

In [ ]:
# trying to pass arguments like so 
def roi_spectra(data, cube, wavelengths):
    '''Compute average spectra for rectangular Regions Of Interest. '''
    
    if not data: # or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([], 'wavelength', 'reflectance')}, kdims='ROI spectra')

    # populate cube dataset 
    # HERE IS THE PROBLEM! 
    h, w, d = cube.shape 
    ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance') 
    
    curves = {}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean)).opts(framewise=True)
    return hv.NdOverlay(curves, kdims='ROI spectra')

def roi_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {i}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

def roi_app(cube, wavelengths, frame_width=500, highres_file=None): 
    '''Simple interactive holoviews based ROI selection of spatial regions of interest for hyperspectral data `cube` 
    
    with `wavelengths` in third dimension. '''
    
    # determine shape etc 
    h, w, d = cube.shape 
    bounds = [0, 0, w, h] 
    pseudo_rgb = cube[:,:, [70, 53, 19]] 
    
    # wrap into holoviews Dataset 
    # for some funny reason the order of dimensions is opposite to the actual numpy order!   
    ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance')

    # initialize Box stream 
    polys = hv.Polygons([])
    box_stream = streams.BoxEdit(source=polys)

    # create interactive plots 
    # The question is here how to combine the dynamic map callback function arguments with the box_stream? 
    spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream], **dict(cube=cube, wavelengths=wavelengths)).opts(frame_width=frame_width, framewise=True)
    #labels_dmap = hv.DynamicMap(roi_labelize, streams=[box_stream]).opts(frame_width=frame_width)
    rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds)).opts(frame_width=frame_width) 

    # combine them into layout
    # NEEDED TO REORDER (COLLATE) THE OVERLAY
    cube_view = hv.Overlay([rgb_view, polys]).collate()# , labels_dmap]).collate()
    roi_picker = hv.Layout([cube_view, spectra_dmap])

    # if available include highres image with identical bounds 
    if highres_file is not None: 
        highres_view = rasterize(RGB.load_image(highres_file, bounds=bounds)).opts(frame_width=frame_width)

        roi_picker = hv.Layout([highres_view, cube_view, spectra_dmap])
    
    return roi_picker 

In [ ]:
frame_width = 700


# determine shape etc 
h, w, d = cube.shape 
bounds = [0, 0, w, h] 
pseudo_rgb = cube[:,:, [70, 53, 19]] 

highres_view = rasterize(RGB.load_image(tif_file, bounds=bounds)).opts(frame_width=frame_width)

# wrap into holoviews Dataset 
# for some funny reason the order of dimensions is opposite to the actual numpy order!   
ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance')

# initialize Box stream 
polys = hv.Polygons([])
box_stream = streams.BoxEdit(source=polys)

# create interactive plots 
spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)
labels_dmap = hv.DynamicMap(roi_labelize, streams=[box_stream]).opts(frame_width=frame_width)
rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds)).opts(frame_width=frame_width) 

# combine them into layout 
roi_pick = rgb_view * polys * labels_dmap + spectra_dmap

In [ ]:
highres_view + roi_pick

In [ ]:
rois_view = roi_app(cube, wavelengths, highres_file=tif_file)

In [ ]:
rois_view

In [ ]:
hv.DynamicMap.__init__??

In [ ]:
rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds, kdims=[f'X{n}', f'Y{n}']))
tif_view = rasterize(RGB.load_image(tif_file, bounds=bounds, kdims=[f'X{n}', f'Y{n}']))
boxes_canvas = hv.Rectangles([])
boxes_stream = streams.BoxEdit(source=boxes_canvas)
boxes_pane = pn.pane.Str(object=boxes_stream.param.data) 
boxes_labels = hv.DynamicMap(box_labelize, streams=[boxes_stream]) 
spectra_dmap = hv.DynamicMap(get_mean_spectra, streams=[boxes_stream])

layout = tif_view + rgb_view * boxes_canvas * boxes_labels + spectra_dmap
pn.Column(layout, boxes_pane)

## Starting with a cube widget selector 

In [ ]:
import panel as pn 

In [ ]:
objnum_widget = pn.widgets.Select(description='Object number', options=obj_nums)

In [ ]:
objnum_widget

Let's study one more time the holoviews tutorial: https://holoviews.org/user_guide/Dashboards.html#creating-interactive-dashboards

In [ ]:
def read_cube(npz_file): 

    npz = np.load(npz_files_dict[num][0])
cube = npz['image'][:,:, ::-1].transpose(1, 2, 0)
wavelengths = npz['wavelengths'] 
h, w, d = cube.shape
bounds = [0, 0, w, h]

In [ ]:
import holoviews as hv 
from holoviews import opts, RGB 
from holoviews.streams import BoxEdit
import param 

opts.defaults(opts.RGB(aspect='equal'))

In [ ]:
boxes = hv.Rectangles([])
boxes_stream = BoxEdit(source=boxes)

In [ ]:
pseudo_view = RGB(pseudo_rgb, bounds=bounds)
txt_pane = pn.pane.Str(boxes_stream.param.data)

In [ ]:
pn.Column(pseudo_view * boxes, txt_pane)

## FUNCTIONS 

In [ ]:
#|export 

import holoviews as hv 
from holoviews import RGB, opts, streams 
from holoviews.operation.datashader import rasterize  
import panel as pn 
import numpy as np 
import toml 
import pandas as pd 
import matplotlib.pyplot as plt
import hvplot.pandas

In [ ]:
#|export 

def roi_spectra(data):
    '''Compute average spectra for rectangular Regions Of Interest. '''
    
    if not data: # or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([], 'wavelength', 'reflectance')}, kdims='ROI spectra')

    curves = {}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean)).opts(framewise=True)
    return hv.NdOverlay(curves, kdims='ROI spectra')

def roi_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {i}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

def roi_picker(cube, wavelengths, frame_width=500, highres_file=None): 
    '''Simple interactive holoviews based ROI selection of spatial regions of interest for hyperspectral data `cube` 
    
    with `wavelengths` in third dimension. '''
    
    # determine shape etc 
    h, w, d = cube.shape 
    bounds = [0, 0, w, h] 
    pseudo_rgb = cube[:,:, [70, 53, 19]] 
    
    # wrap into holoviews Dataset 
    # for some funny reason the order of dimensions is opposite to the actual numpy order!   
    ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance')

    # initialize Box stream 
    polys = hv.Polygons([])
    box_stream = streams.BoxEdit(source=polys)

    # create interactive plots 
    spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)
    #labels_dmap = hv.DynamicMap(roi_labelize, streams=[box_stream]).opts(frame_width=frame_width)
    rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds)).opts(frame_width=frame_width) 

    # combine them into layout
    # NEEDED TO REORDER (COLLATE) THE OVERLAY

    cube_view = hv.Overlay([rgb_view, polys]).collate()# , labels_dmap]).collate()
    roi_picker = hv.Layout([cube_view, spectra_dmap])

    # if available include highres image with identical bounds 
    if highres_file is not None: 
        highres_view = rasterize(RGB.load_image(highres_file, bounds=bounds)).opts(frame_width=frame_width)

        roi_picker = hv.Layout([highres_view, cube_view, spectra_dmap])
    
    return roi_picker 



In [ ]:
# Not sure what went wrong but trying again by exporting the cell above 

hv.extension('bokeh') 
colors = hv.Cycle.default_cycles['default_colors']
boxes_styles = dict(fill_alpha=0.1, line_color='white')
style_opts = dict(size=10, fill_color=colors, line_color='white')
frame_width = 300
opts.defaults(opts.RGB(aspect='equal', frame_width=frame_width, invert_yaxis=False), 
              opts.Image(aspect='equal', frame_width=frame_width), 
              opts.Curve(frame_width=frame_width, frame_height=frame_width, color=hv.Cycle(colors)), 
              opts.Labels(text_color='white'), 
              opts.Rectangles(fill_color=None, line_color='white'), 
              opts.Polygons(fill_color=None, line_color='white'))    

def roi_spectra(data):
    '''Compute average spectra for rectangular Regions Of Interest. '''
    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([], 'Wavelength', 'Reflectance')}, kdims='ROI spectra')

    curves = {}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        curves[f'#{i}'] = hv.Curve(selection.aggregate('Wavelength', np.mean)).opts(framewise=True)
    return hv.NdOverlay(curves, kdims='ROI spectra')

def roi_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        # simple hack to position labels 
        label_list = [[xi, yi, f'    {i}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))]   
        labels = hv.Labels(label_list)

    return labels 

def roi_picker(cube, wavelengths, frame_width=500, highres_file=None): 
    '''Interactive holoviews based selection of spatial regions of interest for hyperspectral data `cube` 
    
    with `wavelengths` in third dimension. '''
    
    # determine shape etc 
    h, w, d = cube.shape 
    bounds = [0, 0, w, h] 
    pseudo_rgb = cube[:,:, [70, 53, 19]] 
    
    # wrap into holoviews Dataset 
    # for some funny reason the order of dimensions is opposite to the actual numpy order!   
    ds = hv.Dataset((wavelengths, np.arange(w), np.arange(h-1, -1, -1), cube), ['Wavelength', 'x', 'y'], 'Reflectance')

    # initialize Box stream 
    polys = hv.Polygons([])
    box_stream = hv.streams.BoxEdit(source=polys)

    # create interactive plots 
    spectra_dmap = hv.DynamicMap(roi_spectra, streams=[box_stream]).opts(frame_width=frame_width, framewise=True)
    labels_dmap = hv.DynamicMap(roi_labelize, streams=[box_stream]).opts(frame_width=frame_width)
    rgb_view = rasterize(RGB(pseudo_rgb, bounds=bounds)).opts(frame_width=frame_width) 

    # combine them into layout 
    roi_picker_view = rgb_view * polys * labels_dmap + spectra_dmap 

    # if available include highres image with identical bounds 
    if highres_file is not None: 
        highres_view = rasterize(RGB.load_image(highres_file, bounds=bounds)).opts(frame_width=frame_width)
        return highres_view + roi_picker_view
    
    return roi_picker_view 

In [ ]:
def box_labelize(data): 
    '''Add labels to boxes stream `data` and return both as a holoviews Overlay.'''
    
    if (data is None) or (len(data['x0']) == 0):
        labels = hv.Labels([])
    else: 
        label_list = [[xi, yi, f'    {i}'] for i, [xi, yi] in enumerate(zip(data['x1'], data['y0']))] # simple hack to position labels  
        labels = hv.Labels(label_list)

    return labels 


def get_mean_spectra(data, cube, nms): 
    '''DynamicMap callback function that returns a holoviews Overlay of 
    spectral Curve plots of mean spectra for all selected rectangles. '''  

    if (data is None):
        overlay = hv.Overlay([hv.Curve(([],[]), kdims='Wavelength!', vdims='Reflectance').opts(xlim=(400, 1000))]).opts(title='Reflectance spectra')

    else: 
        # calculate pseudo rgb image from cube 
        pseudo_rgb = cube[:,:, [70, 53, 19]] 
        h, w, d = cube.shape
    
        # box positions from stream  
        boxes = pd.DataFrame(boxes_stream.data)[['x0', 'x1', 'y0', 'y1']].values
    
        # compute mean spectra for boxes 
        spectra = [] 
        box_colors = []
        for x0, x1, y0, y1 in boxes: 
            
            # invert holoviews y coordinate to numpy indexes!
            roi_w = int(abs(x1 - x0))  
            roi_h = int(abs(y1 - y0)) 
            
            i0 = h - int(max(y0, y1)) 
            i1 = i0 + roi_h
            j0 = int(x0) 
            j1 = j0 + roi_w 

            # compute mean spectrum 
            roi_cube = cube[i0:i1, j0:j1] # cube of interest # check if boundaries ok
            spectrum_table = roi_cube.reshape([-1, d]) 
            spectrum = spectrum_table.mean(axis=0)  
            spectra.append(spectrum) 

            # and color 
            color_table = pseudo_rgb[i0:i1, j0:j1].reshape([-1, 3]) 
            rgb = color_table.mean(axis=0) 
            box_colors.append(rgb) 
                        
        # create holoviews plot with spectra 
        curves = [hv.Curve((nms, s), kdims='Wavelength', vdims='Reflectance', label=f'{i}') for i, s in enumerate(spectra)] #.opts(xlim=(400, 1000), line_color=roi_colors[i]) for i, s in enumerate(mean_spectra)] 
        
        overlay = hv.Overlay(curves).opts(title='Reflectance spectra')

    return overlay 